<a href="https://colab.research.google.com/github/emon4075/PyTorch-For-Deep-Learning/blob/master/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [137]:
# Import the main PyTorch library for neural networks.
import torch
# Import the neural network module from PyTorch.
import torch.nn as nn
# Import matplotlib.pyplot for plotting and visualization.
import matplotlib.pyplot as plt

# Import DataLoader for efficient batching of data.
from torch.utils.data import DataLoader
# Import datasets and transforms from torchvision for common datasets and image transformations.
from torchvision import datasets, transforms

# Set the Device

In [138]:
# Set the device for training (GPU if available, otherwise CPU).
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Image to Tensor

In [139]:
# Define a transformation to convert images to PyTorch tensors. This normalizes pixel values to [0, 1].
transform = transforms.ToTensor()

In [140]:
# Load the MNIST training dataset.
# root='': Specifies the directory to store the dataset.
# train=True: Indicates this is the training split.
# download=True: Downloads the dataset if it's not already present.
# transform=transform: Applies the ToTensor transformation to the images.
train_data = datasets.MNIST(root='', train=True, download=True, transform=transform)

In [141]:
# Load the MNIST test dataset.
# root='': Specifies the directory to store the dataset.
# train=False: Indicates this is the test split.
# download=True: Downloads the dataset if it's not already present.
# transform=transform: Applies the ToTensor transformation to the images.
test_data = datasets.MNIST(root='', train=False, download=True, transform=transform)

In [142]:
# Display information about the training dataset, such as number of datapoints and transformations applied.
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: 
    Split: Train
    StandardTransform
Transform: ToTensor()

In [143]:
# Display information about the test dataset, such as number of datapoints and transformations applied.
test_data

Dataset MNIST
    Number of datapoints: 10000
    Root location: 
    Split: Test
    StandardTransform
Transform: ToTensor()

# Define the CNN

In [144]:
# Define the Convolutional Neural Network (CNN) class.
class CNN(nn.Module):
  # Initialize the CNN model.
  def __init__(self):
    # Call the constructor of the parent class (nn.Module).
    super().__init__()

    # Define the sequential layers of the neural network.
    self.features = nn.Sequential(
        # First Convolutional Layer
        nn.Conv2d(
            in_channels=1,  # Input channels (1 for grayscale images like MNIST).
            out_channels=32, # Number of output channels (feature maps).
            kernel_size=3,  # Size of the convolutional kernel (3x3).
            padding=1       # Add padding to maintain spatial dimensions.
        ),
        nn.ReLU(),      # ReLU activation function for non-linearity.
        nn.MaxPool2d(kernel_size=2), # Max pooling layer with a 2x2 window to reduce spatial dimensions.

        # Second Convolutional Layer
        nn.Conv2d(
            in_channels=32, # Input channels (from the previous layer's output).
            out_channels=64, # Number of output channels.
            kernel_size=3,  # Size of the convolutional kernel.
            padding=1       # Add padding.
        ),
        nn.ReLU(),      # ReLU activation function.
        nn.MaxPool2d(kernel_size=2), # Max pooling layer.

        nn.Flatten(), # Flatten the output of the convolutional layers into a 1D vector.

        # First Fully Connected (Linear) Layer
        # 64*7*7: Input features calculation for a 28x28 MNIST image after two conv/pool layers.
        nn.Linear(64*7*7, 128), # Input features, Output features.
        nn.ReLU(),      # ReLU activation function.
        # Second Fully Connected (Linear) Layer
        nn.Linear(128, 10) # Input features, Output features (10 for 10 classes in MNIST).
    )

  # Define the forward pass of the network.
  def forward(self, x):
    # Pass the input tensor through the defined feature layers.
    return self.features(x)

In [145]:
# Instantiate the CNN model and move it to the specified device (GPU/CPU).
model = CNN().to(device=device)

# Parameters

In [146]:
# Print a summary of the model's parameters. This shows the structure and learnable parameters.
print(model.parameters)

<bound method Module.parameters of CNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=3136, out_features=128, bias=True)
    (8): ReLU()
    (9): Linear(in_features=128, out_features=10, bias=True)
  )
)>


In [147]:
# Define the loss function: CrossEntropyLoss is suitable for multi-class classification problems.
criterion = nn.CrossEntropyLoss()
# Define the optimizer: Adam optimizer is used for updating model weights.
optimizer = torch.optim.Adam(
    lr=0.001, # Set the learning rate for the optimizer.
    params=model.parameters() # Pass all learnable parameters of the model to the optimizer.
)

In [148]:
# Define the batch size for training and testing.
batch_size = 64
# Create a DataLoader for the training data.
# train_data: The dataset to load.
# batch_size: How many samples per batch to load.
# shuffle=True: Shuffles the data at each epoch for better training.
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
# Create a DataLoader for the test data.
# test_data: The dataset to load.
# batch_size: How many samples per batch to load.
# shuffle=False: No need to shuffle test data.
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

# Train the Model

In [149]:
# Set the number of training epochs.
epochs = 10

# Loop through each epoch.
for each in range(epochs):
  # Set the model to training mode.
  model.train()
  # Iterate over batches of images and labels from the training data loader.
  for images, labels in train_loader:
    # Move images and labels to the specified device (GPU/CPU).
    images = images.to(device)
    labels = labels.to(device)
    # Perform a forward pass: get predictions from the model.
    outputs = model(images)
    # Calculate the loss between the predictions and true labels.
    loss = criterion(outputs, labels)

    # Zero the gradients from the previous iteration.
    optimizer.zero_grad()
    # Perform backpropagation: compute gradients of the loss with respect to model parameters.
    loss.backward()
    # Update model parameters using the optimizer.
    optimizer.step()

  # Print the loss after each epoch.
  print(f'Epoch: {each}: Loss -> {loss.item()}')

Epoch: 0: Loss -> 0.05470813065767288
Epoch: 1: Loss -> 0.015412360429763794
Epoch: 2: Loss -> 0.07782717794179916
Epoch: 3: Loss -> 0.0010402417974546552
Epoch: 4: Loss -> 0.0010839991737157106
Epoch: 5: Loss -> 0.11875268816947937
Epoch: 6: Loss -> 0.004282917827367783
Epoch: 7: Loss -> 0.00037644978147000074
Epoch: 8: Loss -> 0.006337199360132217
Epoch: 9: Loss -> 0.0007089904975146055


In [150]:
# Set the model to evaluation mode (disables dropout, batch normalization updates, etc.).
model.eval()
# Initialize variables to track correct predictions and total samples.
correct = 0
total = 0

# Disable gradient calculations during inference to save memory and speed up computation.
with torch.no_grad():
  # Iterate over batches of images and labels from the test data loader.
  for images, labels in test_loader:
    # Move images and labels to the specified device (GPU/CPU).
    images = images.to(device)
    labels = labels.to(device)
    # Perform a forward pass: get predictions from the model.
    outputs = model(images)

    # Get the predicted class by finding the index with the maximum log-probability.
    predictions = torch.argmax(outputs, dim = 1)
    # Count the number of correct predictions in the current batch.
    correct += (predictions == labels).sum().item()
    # Add the total number of labels in the current batch to the overall total.
    total += labels.size(0)

# Calculate the final accuracy.
accuracy = correct / total

# Print the test accuracy formatted to four decimal places.
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.9917
